# H&M 2년 M5 가격구간 오류 사전 점검
같은 사전 점검을 H&M 2년 자료의 기존 M1 체크포인트에 적용합니다. 두 데이터 모두에서 조건을 충족할 때만 가치대 음성 설계로 진행합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = 'a4450d411c4b0fa2f5556e6700255a1c6175ce11'
%cd /content
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA
import subprocess
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() == REVIEWED_SHA


In [ ]:
import importlib
import json
import torch
import lightgcn_clv_m5_price_band_error_diagnostic as band
band = importlib.reload(band)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
assert band.CODE_VERSION == 'm5-price-band-error-diagnostic-v1'
cfg = band.configure_price_band_error_diagnostic('hm')
print(json.dumps(band.preflight_summary(cfg), ensure_ascii=False, indent=2))


In [ ]:
paths = band.run_price_band_error_diagnostic(band.configure_price_band_error_diagnostic('hm'))


In [ ]:
import pandas as pd
from IPython.display import display

summary = pd.read_csv(paths['summary_csv'])
print('1) 가격구간 기준 누락정답–오추천 쌍')
display(summary)
report = json.load(open(paths['json']))
print('2) 판독')
print(json.dumps(report['reading'], ensure_ascii=False, indent=2))
print('판독 기준: 고CLV 같은 구간 쌍 비율 >= 0.30, 그리고 다른 구간 q_V 승률 > 0.5이면서 같은 구간 승률이 0.5에 더 가까울 것 (두 데이터 모두)')
print('결과 파일:', paths)
